<a href="https://colab.research.google.com/github/jensentke/DS-Work-Folder/blob/main/RAG_Part_1_MSDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval Augmented Generation
## Part 1



#Step 0: Install and import useful packages

In [ ]:
pip install PyMuPDF transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 40.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 75.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 94.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [PyMuPDF]


In [ ]:
%pip install nltk

In [ ]:
import os
import fitz  # PyMuPDF
from transformers import AutoTokenizer, AutoModel
import torch
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
import faiss
import numpy as np

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


# Step 1: Read PDF Files

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Example folder path in Google Drive
folder_path = '/content/drive/My Drive/PDFs/'  # Adjust this to your folder path
#file_path = '/content/drive/MyDrive/documents/my_pdf_file.pdf'

def read_pdfs(folder_path):
    pdf_texts = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.pdf'):
            file_path = os.path.join(folder_path, file_name)
            try:
                doc = fitz.open(file_path)
                text = ""
                for page in doc:
                    text += page.get_text()
                pdf_texts.append((file_name, text))
            except Exception as e:
                print(f"Error reading {file_name}: {e}")
    return pdf_texts

# Run the function
pdf_contents = read_pdfs(folder_path)

# Display the results
for file_name, text in pdf_contents:
    print(f"Contents of {file_name}:\n{text[:1000]}...")  # Display first 100 characters for preview

Mounted at /content/drive
Contents of ds_6371_syllabus Ver 7.pdf:
Course Syllabus: DS 6371 Statistical Foundations for Data Science 
 
Course Designers:  
Dr. Bivin Sadler and Dr. Monnie McGee 
Course Text: 
Ramsey, F. L., and D. W. Schafer. The Statistical Sleuth: A Course in 
Methods of Data Analysis, 3rd ed. Boston, MA: Brooks/Cole, 2013, 
with associated website www.statisticalsleuth.com. 
 
Other Materials: 
ChatGPT account … it can be paid or free.   
Prerequisites: 
 
 
A previous introductory statistics course and Bridge to Statistics 
 
Midterm Date 
Saturday, October 19th 2024 from 11am to 2pm CST on Zoom 
Final Exam Date 
Saturday, December 14th from 11am to 2pm CST on Zoom 
The text is available as an electronic version from CengageBrain.com and is much less expensive 
this way! 
All elements of the syllabus are subject to change by the instructor. 
Before taking this class, you should know 
• 
Statistical methods from an introductory statistics course: appropriate use of t

In [ ]:
pdf_texts = read_pdfs(folder_path)

pdf_texts

[('ds_6371_syllabus Ver 7.pdf',
  'Course Syllabus: DS 6371 Statistical Foundations for Data Science \n \nCourse Designers:  \nDr. Bivin Sadler and Dr. Monnie McGee \nCourse Text: \nRamsey, F. L., and D. W. Schafer. The Statistical Sleuth: A Course in \nMethods of Data Analysis, 3rd ed. Boston, MA: Brooks/Cole, 2013, \nwith associated website www.statisticalsleuth.com. \n \nOther Materials: \nChatGPT account … it can be paid or free.   \nPrerequisites: \n \n \nA previous introductory statistics course and Bridge to Statistics \n \nMidterm Date \nSaturday, October 19th 2024 from 11am to 2pm CST on Zoom \nFinal Exam Date \nSaturday, December 14th from 11am to 2pm CST on Zoom \nThe text is available as an electronic version from CengageBrain.com and is much less expensive \nthis way! \nAll elements of the syllabus are subject to change by the instructor. \nBefore taking this class, you should know \n• \nStatistical methods from an introductory statistics course: appropriate use of the mea

# Step 2: Chunk Text

In [ ]:
# Step 2: Chunk Text
def chunk_text(text, chunk_size=100):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        words = sentence.split()
        if current_length + len(words) > chunk_size:
            chunks.append(' '.join(current_chunk))
            current_chunk = []
            current_length = 0
        current_chunk.extend(words)
        current_length += len(words)

    if current_chunk:
        chunks.append(' '.join(current_chunk))

   # Print out each chunk
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i}: {chunk}\n")

    return chunks

In [ ]:
nltk.download('punkt_tab')

all_chunks = []
chunk_mapping = []

for pdf_name, text in pdf_texts:
    chunks = chunk_text(text)
    all_chunks.extend(chunks)
    chunk_mapping.append((pdf_name, chunks))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Chunk 0: Course Syllabus: DS 6371 Statistical Foundations for Data Science Course Designers: Dr. Bivin Sadler and Dr. Monnie McGee Course Text: Ramsey, F. L., and D. W. Schafer. The Statistical Sleuth: A Course in Methods of Data Analysis, 3rd ed. Boston, MA: Brooks/Cole, 2013, with associated website www.statisticalsleuth.com. Other Materials: ChatGPT account … it can be paid or free.

Chunk 1: Prerequisites: A previous introductory statistics course and Bridge to Statistics Midterm Date Saturday, October 19th 2024 from 11am to 2pm CST on Zoom Final Exam Date Saturday, December 14th from 11am to 2pm CST on Zoom The text is available as an electronic version from CengageBrain.com and is much less expensive this way! All elements of the syllabus are subject to change by the instructor.

Chunk 2: Before taking this class, you should know • Statistical methods from an introductory statistics course: appropriate use of the mean and median, interpretation of box plots and histograms, use of

# Step 3: Create Embeddings / Vectorization

In [ ]:
# Step 3: Create Embeddings
def create_embeddings(text_chunks, tokenizer, model):
    embeddings = []
    for chunk in text_chunks:
        inputs = tokenizer(chunk, return_tensors='pt', truncation=True, padding=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        embeddings.append(outputs.last_hidden_state.mean(dim=1).squeeze().numpy())

          # Print out each embedding
    for i, embed in enumerate(embeddings):
        print(f"Embedding {i}: {embed}\n")

    return np.array(embeddings)

In [ ]:
model = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModel.from_pretrained(model)

# Create embeddings
embeddings = create_embeddings(all_chunks, tokenizer, model)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
  1.03604823e-01 -1.11686543e-01 -2.36760348e-01 -1.06743164e-01
  4.91747409e-02 -3.32110710e-02  1.18448194e-02  1.88170031e-01
 -7.65604228e-02 -1.62952736e-01 -1.63556501e-01  2.35964451e-02
 -1.43101498e-01  6.61282390e-02  3.79680358e-02 -6.22262433e-02
 -5.77716865e-02  9.98149216e-02 -1.12141334e-01  2.79248785e-02
  1.86612666e-01 -6.06198870e-02 -5.54291382e-02  1.74472071e-02
 -3.89422067e-02 -9.08003896e-02  7.96811208e-02  1.77816376e-02
  3.53635587e-02  1.03463545e-01  4.89639081e-02 -1.27107184e-02
  1.74538612e-01 -7.81839341e-02  3.89942788e-02 -1.54192701e-01
 -1.69983998e-01  7.66319260e-02  1.00865424e-01 -5.61477058e-02
 -3.46152559e-02 -1.66931271e-03  1.23393780e-03 -3.01890653e-02
 -1.28516674e-01 -1.91318430e-02  1.97975058e-02 -1.18909622e-32
 -1.79553732e-01  1.67325824e-01 -2.65893787e-02  9.18027982e-02
  9.45371240e-02  1.61490351e-01 -4.70670834e-02 -1.24322578e-01
  1.78438827e-01 -1.06936149e-01 -2.136

# Step 4: Index Vectors / Embeddings

Indexing embeddings allows for efficient retrieval of relevant text chunks. Without indexing, finding similar chunks would involve comparing the query embedding against all embeddings, which is computationally expensive.

In [ ]:
# Step 4: Index Embeddings
def index_embeddings(embeddings):
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index


In [ ]:
 # Index embeddings
 index = index_embeddings(embeddings)
 index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x792acb1af630> >

# Step 5: Retrieve and return relevant chunks.
### Note that there is no LLM to provide a refined answer here... we were add this later.

In [ ]:
# Step 5: Answer Questions
def answer_question(question, pdf_texts, index, embeddings, tokenizer, model, top_k=3):
    # Create embedding for the question
    inputs = tokenizer(question, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        question_embedding = model(**inputs).last_hidden_state.mean(dim=1).squeeze().numpy()

    # Search for the nearest text chunks
    _, indices = index.search(np.array([question_embedding]), k=top_k)
    indices = indices[0]

    # Collect top-k chunks
    retrieved_chunks = []
    sources = []
    for idx in indices:
        chunk_offset = idx
        pdf_idx = 0

        while chunk_offset >= len(pdf_texts[pdf_idx][1]):
            chunk_offset -= len(pdf_texts[pdf_idx][1])
            pdf_idx += 1

        pdf_name, chunks = pdf_texts[pdf_idx]
        retrieved_chunks.append(chunks[chunk_offset])
        sources.append(f"{pdf_name}, Chunk {chunk_offset}")



    combined_text = ' '.join(retrieved_chunks)
    return f"Answer: {combined_text}\nSources: {sources}"

In [ ]:
 # Answer question
question = "What percent of the overall grade is the homework grade worth in DS 6371?"
answer = answer_question(question, chunk_mapping, index, embeddings, tokenizer, model, top_k=3)
print(answer)

Answer: The final grade for the course will be calculated on the bases of the earned cumulative percentage and the grade received for each of the components of the cumulative percentage. This course is not graded on a curve. The required cumulative percentage needed to earn each letter grade is given in Table 1. Table 1: Cumulative Percentage Required to Reach Each Letter Grade Cumulative Percentage Earned Grade [100 – 93] A (93 – 90] A- (90 – 88] B+ (88 – 83] B (83 – 80] B- (80 – 78] C+ (78 – 73] C (73 – 70] C- (70 – 60] D < 60 F The cumulative percentage for the course is determined by the course assignment components with their corresponding percentages defined in Table 2. Table 2: Grade Components and Weightings of the Cumulative Percentage Percentage of Cumulative Percentage Component Must complete 100% on time to pass the course. Grading: Midterm exam (25 percent, Points/Scale 0-100), final exam (25 percent, Points/Scale 0-100), homework (10 percent, Points/Scale 0-100), project 

# All Together

In [ ]:
# Step 1: Read PDF Files
def read_pdfs(folder_path):
    pdf_texts = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.pdf'):
            file_path = os.path.join(folder_path, file_name)
            try:
                doc = fitz.open(file_path)
                text = ""
                for page in doc:
                    text += page.get_text()
                pdf_texts.append((file_name, text))
            except Exception as e:
                print(f"Error reading {file_name}: {e}")
    return pdf_texts

# Step 2: Chunk Text
def chunk_text(text, chunk_size=100):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        words = sentence.split()
        if current_length + len(words) > chunk_size:
            chunks.append(' '.join(current_chunk))
            current_chunk = []
            current_length = 0
        current_chunk.extend(words)
        current_length += len(words)

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks


# Step 3: Create Embeddings
def create_embeddings(text_chunks, tokenizer, model):
    embeddings = []
    for chunk in text_chunks:
        inputs = tokenizer(chunk, return_tensors='pt', truncation=True, padding=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        embeddings.append(outputs.last_hidden_state.mean(dim=1).squeeze().numpy())
    return np.array(embeddings)

# Step 4: Index Embeddings
def index_embeddings(embeddings):
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index

# Step 5: Answer Questions
def answer_question(question, pdf_texts, index, embeddings, tokenizer, model, top_k=3):
    # Create embedding for the question
    inputs = tokenizer(question, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        question_embedding = model(**inputs).last_hidden_state.mean(dim=1).squeeze().numpy()

    # Search for the nearest text chunks
    _, indices = index.search(np.array([question_embedding]), k=top_k)
    indices = indices[0]

    # Collect top-k chunks
    retrieved_chunks = []
    sources = []
    for idx in indices:
        chunk_offset = idx
        pdf_idx = 0

        while chunk_offset >= len(pdf_texts[pdf_idx][1]):
            chunk_offset -= len(pdf_texts[pdf_idx][1])
            pdf_idx += 1

        pdf_name, chunks = pdf_texts[pdf_idx]
        retrieved_chunks.append(chunks[chunk_offset])
        sources.append(f"{pdf_name}, Chunk {chunk_offset}")



    combined_text = ' '.join(retrieved_chunks)
    return f"Answer: {combined_text}\nSources: {sources}"


# Main function to tie everything together
def main(folder_path, question, model):
    tokenizer = AutoTokenizer.from_pretrained(model)
    model = AutoModel.from_pretrained(model)

    # Read and chunk PDFs
    pdf_texts = read_pdfs(folder_path)
    all_chunks = []
    chunk_mapping = []

    for pdf_name, text in pdf_texts:
        chunks = chunk_text(text)
        all_chunks.extend(chunks)
        chunk_mapping.append((pdf_name, chunks))

    # Create and index embeddings
    embeddings = create_embeddings(all_chunks, tokenizer, model)
    index = index_embeddings(embeddings)

    # Answer question
    answer = answer_question(question, chunk_mapping, index, embeddings, tokenizer, model)
    print(answer)


# Comparing Different Models

In [ ]:
#question = 'What does the "Check drainage" code mean on the washer?'
#question = 'What is Campus Caring Connections?'
question = "What percent of the overall grade is the homework grade worth in DS 6371?"
#question = "What determines the  largest percent of the grade?"
#question = "What is the FLS assignment?"

__DistilBERT Variants__
  - __distilbert-base-uncased:__ A distilled version of the original BERT model, which is optimized for speed and reduced size, while retaining much of the performance of the larger BERT models.
  - __distilroberta-base:__ A distilled version of the RoBERTa model, offering similar benefits in terms of size and speed.

In [ ]:
main(folder_path, question, 'distilbert-base-uncased')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: • The FLS is critically important to learning in this class and must be completed thoroughly and in a timely manner in order to receive feedback before live session. Discussion Boards: Students are not required to post in discussion boards, unless specified by the professor. Midterm Exam (25 percent, Points/Scale 0-100): There will be a midterm exam in week 8 of the course. It will cover concept and hand-calculation questions, as well as a data analysis question. Please clear your schedule now! We will have a review for the exam during live session 8. Completion of all of the asynchronous video questions is required. In some open-ended questions, everyone can see each other’s answers. This is meant to generate discussion and will be very helpful as a tool to help master the material! “For Live Session” Questions (20 percent, Points/Scale 0-100): The “for live session” assignment is a set of questions and activities that the student is to work on and provide a response to before

In [ ]:
main(folder_path, question, 'distilroberta-base')

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: If you are at all concerned about being prepared for this course … take the Bridge Course … it will help tremendously. The final grade for the course will be calculated on the bases of the earned cumulative percentage and the grade received for each of the components of the cumulative percentage. This course is not graded on a curve. The required cumulative percentage needed to earn each letter grade is given in Table 1. Complete the answers to these questions with at least one slide per question. The idea is that you will present some or all of these in a breakout session during the live session. Make sure to add at least four takeaways on one slide and any questions you have on the last slide. 4. Submit the slide deck to the For Live Session Assignment: Unit X assignment on 2DS by 1pm, Central Time, the day of your live session.
Sources: ['ds_6371_syllabus Ver 7.pdf, Chunk 3', 'ds_6371_syllabus Ver 7.pdf, Chunk 29', 'ds_6371_syllabus Ver 7.pdf, Chunk 25']


__BERT Variants:__
  - __bert-large-uncased:__ A larger version of BERT with more parameters, which can provide better embeddings and improved performance.
  - __roberta-large:__ A robustly optimized BERT approach with more parameters and improved training techniques.

In [ ]:
main(folder_path, question, 'bert-large-uncased')

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-large-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: The final grade for the course will be calculated on the bases of the earned cumulative percentage and the grade received for each of the components of the cumulative percentage. This course is not graded on a curve. The required cumulative percentage needed to earn each letter grade is given in Table 1. Most professors encourage collaborative work except when explicitly prohibited (usually on quizzes and exams). Collaboration means helping one other, not copying answers from one another. Students who turn in exactly the same answers to the same homework will share the grade assigned (i.e., if two students have the same answers, and the grade on the assignment is a 90, then each student will receive a 45). Some instructors may impose stricter penalties. While not ideal, since that unit’s material will not be as thoroughly studied, it may happen. To account for these events, we will drop the lowest homework grade. All data sets for case studies and exercises are available (or wi

__Sentence Transformers:__

  - __all-MiniLM-L6-v2:__ A lightweight model optimized for generating sentence embeddings efficiently.
  - __all-mpnet-base-v2:__ A variant of MPNet optimized for generating high-quality sentence embeddings.

In [ ]:
main(folder_path, question, 'sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Answer: The final grade for the course will be calculated on the bases of the earned cumulative percentage and the grade received for each of the components of the cumulative percentage. This course is not graded on a curve. The required cumulative percentage needed to earn each letter grade is given in Table 1. Table 1: Cumulative Percentage Required to Reach Each Letter Grade Cumulative Percentage Earned Grade [100 – 93] A (93 – 90] A- (90 – 88] B+ (88 – 83] B (83 – 80] B- (80 – 78] C+ (78 – 73] C (73 – 70] C- (70 – 60] D < 60 F The cumulative percentage for the course is determined by the course assignment components with their corresponding percentages defined in Table 2. Table 2: Grade Components and Weightings of the Cumulative Percentage Percentage of Cumulative Percentage Component Must complete 100% on time to pass the course. Grading: Midterm exam (25 percent, Points/Scale 0-100), final exam (25 percent, Points/Scale 0-100), homework (10 percent, Points/Scale 0-100), project 

In [ ]:
main(folder_path, question, 'sentence-transformers/all-mpnet-base-v2')

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Answer: Table 1: Cumulative Percentage Required to Reach Each Letter Grade Cumulative Percentage Earned Grade [100 – 93] A (93 – 90] A- (90 – 88] B+ (88 – 83] B (83 – 80] B- (80 – 78] C+ (78 – 73] C (73 – 70] C- (70 – 60] D < 60 F The cumulative percentage for the course is determined by the course assignment components with their corresponding percentages defined in Table 2. Table 2: Grade Components and Weightings of the Cumulative Percentage Percentage of Cumulative Percentage Component Must complete 100% on time to pass the course. Grading: Midterm exam (25 percent, Points/Scale 0-100), final exam (25 percent, Points/Scale 0-100), homework (10 percent, Points/Scale 0-100), project (10 percent, Points/Scale 0-100), live session attendance (10 percent, Points/Scale 0-100), asynchronous video questions (Students Must Complete 100% of the asynch material on time (by the end of that unit) to pass the course), “for live session” questions (20 percent, Points/Scale 0-100) Homework (10 per